# Notebook 02: Hugging Face Hub

This notebook explores the Hugging Face ecosystem: searching models, reading model cards, loading pipelines, and comparing tokenizer behavior.

**Learning Objectives:**
- Search and discover models on Hugging Face Hub
- Read and interpret model cards
- Load models using the transformers pipeline API
- Understand tokenizer behavior and its impact
- Compare model outputs on the same prompts

## 1. Installing and Importing

In [ ]:
# !pip install huggingface_hub transformers torch sentencepiece

from huggingface_hub import HfApi, list_models, model_info
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import pandas as pd

## 2. Searching Models on Hugging Face

In [ ]:
api = HfApi()

# Search for text generation models
print("=== Top Text Generation Models ===")
models = list(list_models(
    pipeline_tag="text-generation",
    sort="downloads",
    direction=-1,
    limit=10
))

model_data = []
for m in models:
    model_data.append({
        "Model": m.id,
        "Downloads": m.downloads if hasattr(m, 'downloads') else "N/A",
        "Likes": m.likes if hasattr(m, 'likes') else "N/A",
        "Pipeline": m.pipeline_tag
    })

display(pd.DataFrame(model_data))

In [ ]:
# Search with filters
print("=== Text Generation Models (English, MIT/Apache License) ===")
filtered_models = list(list_models(
    pipeline_tag="text-generation",
    language="en",
    sort="downloads",
    direction=-1,
    limit=10
))

for m in filtered_models:
    license_type = getattr(m, 'license', 'Unknown')
    print(f"  {m.id} | License: {license_type}")

## 3. Reading Model Cards

In [ ]:
# Get detailed info about a specific model
model_id = "mistralai/Mistral-7B-Instruct-v0.3"

info = model_info(model_id)

print(f"Model: {info.id}")
print(f"Downloads: {info.downloads}")
print(f"Likes: {info.likes}")
print(f"Library: {info.library_name}")
print(f"Tags: {info.tags}")
print(f"Pipeline tag: {info.pipeline_tag}")

# Check model card content
if hasattr(info, 'card_data') and info.card_data:
    print(f"\nModel card data:")
    for key, value in info.card_data.items():
        print(f"  {key}: {value}")

In [ ]:
# Compare model cards for multiple models
models_to_compare = [
    "mistralai/Mistral-7B-Instruct-v0.3",
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "microsoft/Phi-3.5-mini-instruct"
]

comparison = []
for mid in models_to_compare:
    try:
        info = model_info(mid)
        comparison.append({
            "Model": mid.split("/")[1],
            "Downloads": info.downloads,
            "Likes": info.likes,
            "Library": info.library_name,
            "Tags": str(info.tags[:5]) if info.tags else "N/A"
        })
    except Exception as e:
        comparison.append({"Model": mid, "Error": str(e)})

display(pd.DataFrame(comparison))

## 4. Loading Models with Transformers Pipeline

In [ ]:
# Load a small text generation model
# Using TinyLlama for fast loading (adjust if you have more GPU/memory)
print("Loading TinyLlama-1.1B-Chat...")
generator = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype="auto",
    device_map="auto"
)
print("Model loaded!")

In [ ]:
# Generate text
prompts = [
    "What is machine learning?",
    "Write a Python function to calculate factorial.",
    "Explain the difference between TCP and UDP."
]

for prompt in prompts:
    messages = [
        {"role": "user", "content": prompt}
    ]
    
    output = generator(
        messages,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
        top_p=0.9
    )
    
    print(f"Prompt: {prompt}")
    print(f"Response: {output[0]['generated_text'][-1]['content']}")
    print("-" * 60)

## 5. Tokenizer Usage

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

# Test tokenization
test_texts = [
    "Hello, how are you?",
    "The quick brown fox jumps over the lazy dog.",
    "antidisestablishmentarianism",
    "def fibonacci(n):\n    if n <= 1:\n        return n\n    return fibonacci(n-1) + fibonacci(n-2)"
]

for text in test_texts:
    tokens = tokenizer.encode(text)
    decoded = [tokenizer.decode([t]) for t in tokens]
    print(f"Text: {text[:50]}{'...' if len(text) > 50 else ''}")
    print(f"Token count: {len(tokens)}")
    print(f"Tokens: {decoded}")
    print()

In [ ]:
# Compare tokenization across different tokenizers
tokenizers_to_compare = [
    ("TinyLlama (SentencePiece)", "TinyLlama/TinyLlama-1.1B-Chat-v1.0"),
]

# Add tiktoken (GPT-4 tokenizer) if available
try:
    import tiktoken
    gpt_tokenizer = tiktoken.encoding_for_model("gpt-4")
    has_gpt = True
except:
    has_gpt = False

comparison_text = "The quick brown fox jumps over the lazy dog."

print(f"Text: {comparison_text}\n")

for name, tok_id in tokenizers_to_compare:
    tok = AutoTokenizer.from_pretrained(tok_id)
    tokens = tok.encode(comparison_text)
    print(f"{name}: {len(tokens)} tokens -> {[tok.decode([t]) for t in tokens]}")

if has_gpt:
    tokens = gpt_tokenizer.encode(comparison_text)
    print(f"GPT-4 (tiktoken): {len(tokens)} tokens -> {[gpt_tokenizer.decode([t]) for t in tokens]}")

## 6. Comparing Model Outputs

Send the same prompts to different models and compare.

In [ ]:
# Load a second model for comparison
# Uncomment and change model ID to compare different models

# print("Loading Microsoft Phi-3.5-mini...")
# generator_phi = pipeline(
#     "text-generation",
#     model="microsoft/Phi-3.5-mini-instruct",
#     torch_dtype="auto",
#     device_map="auto"
# )
# print("Phi-3.5 loaded!")

In [ ]:
# Compare outputs on the same prompts
compare_prompts = [
    "What are the benefits of microservices architecture?",
    "Write a SQL query to find the top 5 customers by total order value."
]

for prompt in compare_prompts:
    print(f"Prompt: {prompt}")
    print(f"\n--- TinyLlama Output ---")
    
    messages = [{"role": "user", "content": prompt}]
    output = generator(messages, max_new_tokens=250, temperature=0.3)
    print(output[0]['generated_text'][-1]['content'])
    
    # Uncomment to compare with another model:
    # print(f"\n--- Phi-3.5 Output ---")
    # output_phi = generator_phi(messages, max_new_tokens=250, temperature=0.3)
    # print(output_phi[0]['generated_text'][-1]['content'])
    
    print("\n" + "=" * 60 + "\n")

## 7. Other Pipeline Tasks

In [ ]:
# Sentiment Analysis
print("=== Sentiment Analysis ===")
sentiment = pipeline("sentiment-analysis")

texts = [
    "I love this product! It's amazing.",
    "This is the worst experience I've ever had.",
    "The movie was okay, nothing special."
]

for text in texts:
    result = sentiment(text)
    print(f"  {text[:50]}... -> {result[0]['label']} ({result[0]['score']:.3f})")

In [ ]:
# Text Summarization
print("=== Text Summarization ===")
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

article = """
Artificial intelligence has made significant strides in recent years, with large language 
models (LLMs) becoming increasingly capable. These models, trained on vast amounts of 
text data, can generate human-like text, answer questions, write code, and perform 
various reasoning tasks. Companies like OpenAI, Google, Anthropic, and Meta are 
competing to build the most capable models. However, concerns about AI safety, 
bias, and the environmental cost of training these models continue to grow. 
Researchers are working on more efficient architectures and training methods to 
address these challenges while pushing the boundaries of what AI can achieve.
"""

result = summarizer(article, max_length=60, min_length=20)
print(f"Original length: {len(article.split())} words")
print(f"Summary: {result[0]['summary_text']}")

## Key Takeaways

1. **Hugging Face Hub** is the central repository for open-source ML models — learn to search and filter effectively
2. **Model cards** document capabilities, limitations, and usage — always read them before choosing a model
3. **Pipelines** provide a simple API for common tasks — quick way to test models without boilerplate
4. **Tokenizers** vary between models and directly impact cost — compare tokenization efficiency
5. **Small models** (TinyLlama, Phi-3.5) are surprisingly capable for many tasks and run on consumer hardware